# Image Region of Interest Selection (aka Masking)

In this tutorial, you will learn about how to select regions of interest as well as how to create and apply image masks based on pixel values. These two concepts are actually the same in that they involve flagging sets of pixels as bad. The main difference is semantic
* A region of interest is just a way to flag pixels as good/bad based on coordinate values, rather than pixel values. For example, selecting circular, rectangular, and polygonal areas are region selections.

* Masking is a more general term, often used to differentiate from regions by meaning flagging pixels as bad based on pixel values. So, creating a mask based on where image pixel values are > 0.5 is an example of masking.

Flagging is the general term that includes both selecting regions of interest (based on coordinate values) and masking (based on pixel values).

Flags can be applied to an image by either specifying an expression describing how to choose good pixels or by applying
an existing set of flags. In addition, there is support for applying CRTF (Casa Region Text Format) strings and files to specify region selection. This support is currently limited to
pixel coordinate specification, with support for world coordinates to come after the xradio image schema definition stablizes. 

This demo provides examples of all these methods.

**Note:** This notebook covers selection on generic arrays using pixel (integer-index) coordinates only.
For xradio-format images — where shapes can be specified in angular offsets (arcsec/arcmin)
or absolute sky coordinates (RA/Dec), and where frequency, polarization, and time ranges
can also be applied — see **`xradio_image_selection.ipynb`** in the same directory.

In [ ]:
# imports
from pathlib import Path

import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from astroviper.distributed_applications.image_analysis.selection import (
    apply_select,
    combine_with_creation,
    select_mask,
)
from astroviper.distributed_applications.model.component_models import make_gauss2d

We start by simulating a source based on a 2d elliptical gaussian model with a uniform additive level across the image. We will use xr.DataArrays for the most part, however, there is similar support for flagging for dask and numpy arrays.

In [ ]:
nx, ny = 160, 128
x, y = np.mgrid[0:nx, 0:ny]
data = np.zeros([nx, ny])
xda = xr.DataArray(
    data,
    dims=("x", "y"),
    coords={"x": np.arange(nx), "y": np.arange(ny)},
    name="intensity",
)
img = 0.1 + make_gauss2d(
    data=xda, a=27.0, b=18.0, x0=nx / 2, y0=ny / 2, theta=0.0, peak=1.0
)

img

Plot the model source

In [ ]:
# define plotting helper

ArrayLike = xr.DataArray | np.ndarray | da.core.Array
ARRAY_TYPES = (xr.DataArray, np.ndarray, da.core.Array)


def plot(
    data: ArrayLike | list[ArrayLike],
    title: str = None,
    alpha: float | int | list[float] | list[int] = None,
):
    arys = [data] if isinstance(data, ARRAY_TYPES) else data
    alphas = [alpha] * len(arys) if isinstance(alpha, int | float) else alpha

    fig, ax = plt.subplots()

    for i, ary in enumerate(arys):
        a = None if alpha is None else alphas[i]
        # wrap if necessary to take advantage of
        # xr.DataArray.plot()
        ar = (
            ary
            if isinstance(ary, xr.DataArray)
            else xr.DataArray(ary, dims=("x", "y"), name="")
        )
        img = ar.plot(x="x", y="y", ax=ax, alpha=a, add_colorbar=(i == 0))
        if i == 0 and getattr(img, "colorbar", None) is not None:
            img.colorbar.set_label("")

    ax.set_aspect("equal")
    if title:
        ax.set_title(title)
    plt.show()


plot(img, title="Source")

## Primary methods

Use the **apply_select()** method to apply an expression to an image.
Use **select_mask()** to generate a boolean array that can saved as a data variable and/or applied to an image with **apply_select()**

## **select=None** → use everything, all pixels are good.
Not surprisingly, the plot is identical to the previous one.

In [ ]:
sel_all = apply_select(img, select=None)
plot(sel_all, title="Source With No Selection Applied")

To generate a boolean flag that can be applied to an array later, use **select_mask()**. Because there is no selection in this case,
the mask values are all True; all pixels are good.

In [ ]:
no_mask = select_mask(img, select=None)
no_mask

## Boolean array-like (DataArray or ndarray selection)
Create a circular region of interest (ROI) mask and apply it.

In [ ]:
cx, cy = nx / 2, ny / 2
r = min(nx, ny) / 3.2  # why: focus on core region
# create the boolean region of interest. good pixels are inside the circle. x and y come from the grid we initially created previously.
roi = ((x - cx) ** 2 + (y - cy) ** 2) <= (r**2)

roi_da = xr.DataArray(roi, dims=("x", "y"), coords=xda.coords, name="roi")

In [ ]:
# apply the region of interest to the image to
# create a new image with flagged pixels. Pixels
# outside the roi are marked as bad (nan).
sel_roi = apply_select(img, select=roi_da)
print("bad pixel value", sel_roi[0, 0].values)

In [ ]:
# plot the applied region of interest
plot([img, roi_da], title="Image and Circular Region of Interest", alpha=[1.0, 0.3])
plot(sel_roi, title="Image Created by Applying Region of Interest")

## Build masks from combinations of other masks
One can build a mask using a composite of other masks.

To illustrate this, first create a mask, "good", based on pixel value threshholding

In [ ]:
good = xr.DataArray(np.random.random((nx, ny)) > 0.05, dims=("x", "y"))
expr = "good"
plot(good, title=f"Expression mask: {expr}")

Now, create a region of interest that is confined
to a vertical strip near the image center.

In [ ]:
good_strip = xr.DataArray((x > nx / 2 - 5) & (x < nx / 2 + 5), dims=("x", "y"))
expr = "good strip"
plot(good_strip, title=f"Expression mask: {expr}")

Now combine the circular region, the threshholded mask, and the image center strip in an expression represented by a string.

In [ ]:
# first construct a dictionary with the masks

mask_source = {"roi": roi_da, "good": good, "good_strip": good_strip}
expr = "roi & good & good_strip"
mask_expr = select_mask(img, select=expr, mask_source=mask_source)
# the benefit to specifying mask_source rather than simply using
# an equivalent python expression to construct the mask is that an
# attribute is added to the output with information regarding how
# the mask was created, thus providing an audit trail of sorts
print("creation attr of the mask:", mask_expr.attrs["creation"])
plot([img, mask_expr], title=f"Image and Mask {expr}", alpha=[1.0, 0.3])

In [ ]:
# plot the resulting image that has the expression
# mask applied
sel_expr = apply_select(img, select=expr, mask_source=mask_source)
plot(sel_expr, f"Image with applied selection {expr}")

## CRTF (CASA 6) region support.

Data: simple image with a bright blob

In [ ]:
nx = ny = 200
ary = xr.DataArray(np.zeros([nx, ny]), dims=("x", "y"))
x, y = np.mgrid[0:nx, 0:ny]
z = 0.1 + make_gauss2d(
    ary,
    x0=60.0,
    y0=120.0,
    a=15.0,
    b=15.0,
    theta=0,
    peak=3.0,
)
plot(z, title="Original image (pixel coords)")

Utility to overlay masks

In [ ]:
def show_mask(ax, data: xr.DataArray, mask: xr.DataArray, title: str):
    # base image
    ax.imshow(data, origin="lower", interpolation="none")
    # dim the outside region
    ax.imshow(~mask, origin="lower", interpolation="none", alpha=0.18)
    # highlight the kept region
    ax.imshow(
        np.ma.masked_where(~mask, data),
        origin="lower",
        interpolation="none",
        alpha=0.85,
    )
    # draw a crisp boundary
    ax.contour(mask, levels=[0.5], colors="k", linewidths=1.2, origin="lower")
    ax.set_title(title)

### Box region
CRTF header is optional — both forms are supported. Currently only pixel coordinates are supported.
Pixel coordinate values are interpreted as being 0-based with
origin in the lower left corner.

Syntax `box[[blc_x, blc_y], [trc_x, trc_y]]`

In [ ]:
crtf_no_header = "box[[ 30pix, 70pix ], [ 90pix, 170pix ]]"
crtf_with_header = f"""
#CRTF
{crtf_no_header}
""".strip()
mask1 = select_mask(z, select=crtf_with_header)
mask1b = select_mask(z, select=crtf_no_header)
desc = "One has CRTF header, one doesn't. They should look identical"
# As before the masks have a creation attr
print(
    "creation attrs. The only difference should be that one has "
    "the CRTF header and the other doesn't"
)
print("creation attr of one mask", mask1.attrs["creation"])
print("creation attr of the other mask", mask1b.attrs["creation"])

In [ ]:
print(desc)
plot([z, mask1], title="box with #CRTF header", alpha=[1.0, 0.3])
plot([z, mask1b], title="box without #CRTF header", alpha=[1.0, 0.3])

Apply the regions.

In [ ]:
sel = apply_select(z, select=mask1)
selb = apply_select(z, select=mask1b)
print(desc)
plot(sel, title="box with #CRTF header")
plot(selb, title="box without #CRTF header")

### Rotated box (rotbox)
Syntax: `rotbox[[center_x,center_y],[width,height], mode=angle]`; angle in `deg` or `rad` (default = deg).

The angle can be measured in one of two ways (modes). **mode=theta_m** means use convention where the angle is measured from the +x axis to the +y axis. **pa** means measure from the +y axis to the +x axis. Both are coordinate system handedness agnostic. The specific mode keyword is required to avoid ambiguity.

In this example, we use theta_m as the mode keyword. The angle is thus measured from +x -> +y.

In [ ]:
rot = "rotbox[[75pix,125pix],[60pix,30pix], theta_m=30deg]"
mask_rot = select_mask(z, select=rot)
plot(mask_rot, title=rot)

In [ ]:
sel = apply_select(z, select=mask_rot)
plot(sel, rot)

Here we use the mode of pa, so that the rotation angle is measured from +y -> +x

In [ ]:
rot = "rotbox[[75pix,125pix],[60pix,30pix], pa=30deg]"
mask_rot_pa = select_mask(z, select=rot)
plot(mask_rot_pa, title=rot)

In [ ]:
sel = apply_select(z, select=mask_rot_pa)
plot(sel, title="rot")

### Circle and annulus
syntax `circle[[center_x, center_y], radius]`
`annulus[[center_x, center_y], [inner_radius, outer_radius]]`


In [ ]:
circle = "circle[[75pix,125pix], 35pix]"
annulus = "annulus[[75pix,125pix], [20pix, 50pix]]"
mask_circle = select_mask(z, select=circle)
mask_ann = select_mask(z, select=annulus)
plot(mask_circle, title=circle)
plot(mask_ann, title=annulus)

In [ ]:
sel_circle = apply_select(z, select=mask_circle)
sel_ann = apply_select(z, select=mask_ann)

plot(sel_circle, title=circle)
plot(sel_ann, title=annulus)

### Ellipse with position angle
Syntax: `ellipse[[center_x,center_y], [a,b], pa]` where `a`, `b` are **semi-axes**.

As with rotbox, the angle convention must be specified.

The first example uses theta_m (+x -> +y)

In [ ]:
ellipse = "ellipse[[75pix,125pix], [50pix, 25pix], theta_m=60]"  # 60 degrees
mask_ellipse = select_mask(z, select=ellipse)
plot(mask_ellipse, title=ellipse)

In [ ]:
sel_ellipse = apply_select(z, select=mask_ellipse)
plot(sel_ellipse)

In the following example, we use pa rotation angle convention (+y -> +x)

In [ ]:
ellipse_pa = "ellipse[[75pix,125pix], [50pix, 25pix], pa=60]"  # 60 degrees
mask_ellipse_pa = select_mask(z, select=ellipse_pa)
plot(mask_ellipse_pa, title=ellipse_pa)

In [ ]:
sel_ellipse_pa = apply_select(z, select=mask_ellipse_pa)
plot(sel_ellipse_pa, title=ellipse_pa)

### Polygon (poly)
Syntax `poly[[x0, y0], [x1, y1], ... [xn,yn]]`

In [ ]:
poly = "poly[[40pix,40pix],[70pix,60pix],[60pix,150pix],[30pix,80pix]]"
mask_poly = select_mask(z, select=poly)
plot(mask_poly, title=poly)

In [ ]:
sel_poly = apply_select(z, select=mask_poly)
plot(sel_poly, title=poly)

## Multiple CRTF specifications with `+` (OR) and `-` (subtract)
Lines are combined; default is `+` if no prefix.

In [ ]:
multi = """
#CRTF
+circle[[120pix,120pix], 40pix]
-rotbox[[110pix,145pix],[35pix,20pix], theta_m=25deg]
+box[[20pix,120pix],[100pix,190pix]]
""".strip()

mask_multi = select_mask(z, select=multi)
plot(mask_multi, "multi region")

And the creation attr

In [ ]:
print(f"Multi region creation attr\n{mask_multi.attrs['creation']}")

Apply selection to the image and compute simple stats

In [ ]:
sel_multi = apply_select(z, select=multi)
plot(sel_multi, title="Mulipte CRTF")

### Mixing CRTF with named-mask expressions
If the string **is not** detected as CRTF, it is treated as a named-mask expression.

This example uses `roi & edge` combining masks provided in a mapping or dataset. The
& means include only pixels that are present in both selections; it is an
intersection operator.

In [ ]:
roi = select_mask(z, select="circle[[120pix,80pix], 60pix]")
edge = select_mask(z, select="box[[1pix,1pix],[200pix,30pix]]") | select_mask(
    z, select="box[[270pix,1pix],[170pix,200pix]]"
)
mask_source = {"roi": roi, "edge": edge}
expr = "roi & edge"
mask_expr = select_mask(z, select=expr, mask_source=mask_source)
print()
plot([z, mask_expr], title=mask_expr.attrs["creation"], alpha=[1.0, 0.3])

In [ ]:
sel_expr = apply_select(z, select=mask_expr)
plot(sel_expr, title=mask_expr.attrs["creation"])

Here we use `roi & -edge`. Note the minus sign which negates the selection; **edge** is excluded from the region selection. The order of operations is that `-` is executed prior to `&`, or in general before any combination operator.

In [ ]:
expr = "roi & ~edge"
mask_expr = select_mask(z, select=expr, mask_source=mask_source)
plot([z, mask_expr], title=mask_expr.attrs["creation"], alpha=[1.0, 0.3])

In [ ]:
sel_expr = apply_select(z, select=mask_expr, mask_source=mask_source)
plot(sel_expr, title=mask_expr.attrs["creation"])

Here we OR (|) the regions together, which results in the union of the two regions

In [ ]:
expr = "roi | edge"
mask_expr = select_mask(z, select=expr, mask_source=mask_source)
plot([z, mask_expr], title=mask_expr.attrs["creation"], alpha=[1.0, 0.3])

In [ ]:
sel_expr = apply_select(z, select=mask_expr)
plot(sel_expr, title=mask_expr.attrs["creation"])

Here we create the union of roi and the negation of edge.

In [ ]:
expr = "roi | ~edge"
mask_expr = select_mask(z, select=expr, mask_source=mask_source)
plot([z, mask_expr], title=mask_expr.attrs["creation"], alpha=[1.0, 0.3])

In [ ]:
sel_expr = apply_select(z, select=mask_expr)
plot(sel_expr, title=mask_expr.attrs["creation"])

## Using CRTF **files** with `select_mask`
CRTF files can be referenced via **backticked strings** (e.g. ``"`regions/roi.crtf`"``) or a `pathlib.Path` object.

### Write a CRTF file to disk
The header `#CRTF` is optional.

In [ ]:
regions_dir = Path("regions")
regions_dir.mkdir(exist_ok=True)
crtf_path = regions_dir / "roi.crtf"
crtf_text = """
#CRTF
+circle[[50pix,130pix], 40pix]
-rotbox[[50pix,130pix],[36pix,18pix], theta_m=30deg]
+box[[20pix,160pix],[70pix,190pix]]
""".strip()
crtf_path.write_text(crtf_text, encoding="utf-8")
print("Wrote:", crtf_path)

### Two supported ways to load the CRTF file
1) Backticked string literal **inside** a normal Python string.
2) `Path` object.

In [ ]:
mask_file_bt = select_mask(z, select=f"`{crtf_path.as_posix()}`")
mask_file_p = select_mask(z, select=crtf_path)
assert (
    isinstance(mask_file_bt, xr.DataArray)
    and isinstance(mask_file_p, xr.DataArray)
    and (mask_file_bt == mask_file_p).all()
), "Backtick and Path forms must produce identical masks"

### Compare against the same CRTF provided inline as a string

In [ ]:
mask_inline = select_mask(z, select=crtf_text)
assert (
    isinstance(mask_inline, xr.DataArray)
    and isinstance(mask_file_p, xr.DataArray)
    and (mask_inline == mask_file_p).all()
), "Inline and Path forms must produce identical masks"

### Apply the selection to the image

In [ ]:
sel = apply_select(z, select=f"`{crtf_path.as_posix()}`")
plot(sel, "ROI from CRTF file")

## Selection masks: inputs × outputs

Quick tour of `select_mask(..., return_kind=...)` with NumPy / xarray (NumPy + Dask) inputs, CRTF strings & expressions, and all four output kinds:  
`"numpy"`, `"dask"`, `"dataarray-numpy"`, `"dataarray-dask"` (default).


### Helpers

In [ ]:
def describe(name, arr):
    kind = type(arr).__name__
    if isinstance(arr, xr.DataArray):
        backing = type(arr.data).__name__
        chunks = getattr(arr.data, "chunks", None)
        print(f"{name}: xr.DataArray<{backing}>, shape={arr.shape}, chunks={chunks}")
    else:
        chunks = getattr(arr, "chunks", None)
        print(f"{name}: {kind}, shape={np.shape(arr)}, chunks={chunks}")

### Create sample data (NumPy, DataArray[NumPy], DataArray[Dask])

In [ ]:
nx, ny = 120, 160
data_np = np.zeros((nx, ny), dtype=float)
data_dask = da.zeros((nx, ny), chunks=(40, 40))
data_da_np = xr.DataArray(np.zeros((nx, ny), dtype=float), dims=("x", "y"))
data_da_dask = xr.DataArray(da.zeros((nx, ny), chunks=(40, 40)), dims=("x", "y"))

describe("data_np", data_np)
describe("data_dask", data_dask)
describe("data_da_np", data_da_np)
describe("data_da_dask", data_da_dask)

### Define selections: CRTF (pixel) and expressions

In [ ]:
## Pixel CRTF strings (0-based with 'pix' units)
crtf_box = """
#CRTF
box[[20pix, 25pix], [90pix, 80pix]]
""".strip()

crtf_circle = "circle[[60pix,60pix], 30pix]"

# Named masks for expressions
roi = select_mask(data_dask, select=crtf_box, return_kind="dataarray-numpy")
bad = xr.DataArray(np.zeros((nx, ny), dtype=int), dims=("x", "y"))
bad.values[10:20, 10:30] = 1

expr = "roi | bad"
mask_src = {"roi": roi, "bad": bad}

### NumPy data → all return kinds (CRTF example)

In [ ]:
"""
x = select_mask(data_np, select=expr, mask_source=mask_src, return_kind="numpy")
plot((data_dask,x), alpha=(1.0,0.3) )
"""
None

In [ ]:
m_np_numpy = select_mask(data_np, select=crtf_circle, return_kind="numpy")
m_np_dask = select_mask(
    data_np, select=crtf_circle, return_kind="dask", dask_chunks=(40, 40)
)
m_np_danp = select_mask(data_np, select=crtf_circle, return_kind="dataarray-numpy")
m_np_dadask = select_mask(
    data_np, select=crtf_circle, return_kind="dataarray-dask", dask_chunks=(40, 40)
)

describe("m_np_numpy", m_np_numpy)
describe("m_np_dask", m_np_dask)
describe("m_np_danp", m_np_danp)
describe("m_np_dadask", m_np_dadask)

plot([data_np, m_np_numpy], "NumPy Mask", [1.0, 0.3])
plot([data_np, m_np_dask], "Dask Mask", [1.0, 0.3])
plot([data_np, m_np_danp], "DataArray[NumPy]", [1.0, 0.3])
plot([data_np, m_np_dadask], "DataArray[Dask]", [1.0, 0.3])

### Dask data → all return kinds (CRTF example)

In [ ]:
m_dask_numpy = select_mask(data_dask, select=crtf_circle, return_kind="numpy")
m_dask_dask = select_mask(
    data_dask, select=crtf_circle, return_kind="dask", dask_chunks=(40, 40)
)
m_dask_danp = select_mask(data_dask, select=crtf_circle, return_kind="dataarray-numpy")
m_dask_dadask = select_mask(
    data_dask, select=crtf_circle, return_kind="dataarray-dask", dask_chunks=(40, 40)
)

describe("m_dask_numpy", m_dask_numpy)
describe("m_dask_dask", m_dask_dask)
describe("m_dask_danp", m_dask_danp)
describe("m_dask_dadask", m_dask_dadask)

plot([data_dask, m_dask_numpy], "NumPy Mask", [1.0, 0.3])
plot([data_dask, m_dask_dask], "Dask Mask", [1.0, 0.3])
plot([data_dask, m_dask_danp], "DataArray[NumPy]", [1.0, 0.3])
plot([data_dask, m_dask_dadask], "DataArray[Dask]", [1.0, 0.3])

### DataArray[NumPy] data → all return kinds (expression example)

In [ ]:
m_danp_numpy = select_mask(
    data_da_np, select=expr, mask_source=mask_src, return_kind="numpy"
)
m_danp_dask = select_mask(
    data_da_np, select=expr, mask_source=mask_src, return_kind="dask"
)
m_danp_danp = select_mask(
    data_da_np, select=expr, mask_source=mask_src, return_kind="dataarray-numpy"
)
m_danp_dadask = select_mask(
    data_da_np, select=expr, mask_source=mask_src, return_kind="dataarray-dask"
)

describe("m_danp_numpy", m_danp_numpy)
describe("m_danp_dask", m_danp_dask)
describe("m_danp_danp", m_danp_danp)
describe("m_danp_dadask", m_danp_dadask)

plot([data_da_np, m_danp_numpy], "NumPy Mask", [1.0, 0.3])
plot([data_da_np, m_danp_dask], "Dask Mask", [1.0, 0.3])
plot([data_da_np, m_danp_danp], "DataArray[NumPy]", [1.0, 0.3])
plot([data_da_np, m_danp_dadask], "DataArray[Dask]", [1.0, 0.3])

### DataArray[Dask] data → all return kinds (CRTF example)

In [ ]:
m_dadask_numpy = select_mask(data_da_dask, select=crtf_box, return_kind="numpy")
m_dadask_dask = select_mask(data_da_dask, select=crtf_box, return_kind="dask")
m_dadask_danp = select_mask(
    data_da_dask, select=crtf_box, return_kind="dataarray-numpy"
)
m_dadask_dadask = select_mask(
    data_da_dask, select=crtf_box, return_kind="dataarray-dask"
)

describe("m_dadask_numpy", m_dadask_numpy)
describe("m_dadask_dask", m_dadask_dask)
describe("m_dadask_danp", m_dadask_danp)
describe("m_dadask_dadask", m_dadask_dadask)

plot([data_da_dask, m_np_numpy], "NumPy Mask", [1.0, 0.3])
plot([data_da_dask, m_np_dask], "Dask Mask", [1.0, 0.3])
plot([data_da_dask, m_np_danp], "DataArray[NumPy]", [1.0, 0.3])
plot([data_da_dask, m_np_dadask], "DataArray[Dask]", [1.0, 0.3])

### Applying selections to data

In [ ]:
sel_np = apply_select(data_np, select=expr, mask_source=mask_src)
sel_dask = apply_select(data_dask, select=expr, mask_source=mask_src)

sel_da_np = apply_select(data_da_np, select=expr, mask_source=mask_src)
sel_da_dask = apply_select(data_da_dask, select=expr, mask_source=mask_src)

plot(sel_np, "Applied to NumPy")
plot(sel_dask, "Applied to Dask")
plot(sel_da_np, "Applied to DataArray[NumPy]")
plot(sel_da_dask, "Applied to DataArray[Dask]")

### CRTF file usage: backticked path and `Path`

In [ ]:
crtf_text = """
#CRTF
# an example file
rotbox[[80pix,60pix],[40pix,20pix], theta_m=30]
poly[[20pix,20pix],[40pix,30pix],[30pix,50pix]]
""".strip()

crtf_path = Path("example.crtf")
_ = crtf_path.write_text(crtf_text, encoding="utf-8")

m_inline = select_mask(data_da_np, select=crtf_text)
m_bt = select_mask(data_da_np, select=f"`{crtf_path.as_posix()}`")
m_path = select_mask(data_da_np, select=crtf_path)

describe("m_inline", m_inline)
describe("m_bt", m_bt)
describe("m_path", m_path)
"""
fig, axs = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
show_mask(axs[0], data_da_np, m_inline, "inline text")
show_mask(axs[1], data_da_np, m_bt, "backticked file")
show_mask(axs[2], data_da_np, m_path, "Path object")
plt.show()
"""
plot((data_da_np, m_inline), "Inline CRTF", (1.0, 0.3))
plot((data_da_np, m_bt), "Backticked CRTF filename", (1.0, 0.3))
plot((data_da_np, m_path), "Path(CRTF filename)", (1.0, 0.3))

### Laziness and compute()

Returned dask masks are lazy

In [ ]:
m_lazy = select_mask(
    data_da_dask, select=crtf_circle, return_kind="dask", dask_chunks=(40, 40)
)
print("lazy?", hasattr(m_lazy, "compute"))
m_eager = m_lazy.compute()

describe("m_lazy", m_lazy)
describe("m_eager", m_eager)

### Selection “creation” attribute demo

As mentioned above, `select_mask` attaches a `creation` attribute to **xarray.DataArray** masks, preserving the string (CRTF or expression) or file reference used to create the mask. Shown here is also a tiny helper to preserve/merge `creation` when you combine masks with bitwise ops.

In [ ]:
# Make a sample image (xarray, coordinates optional)
ny, nx = 200, 300
xda = xr.DataArray(np.zeros((ny, nx), dtype=float), dims=("x", "y"))

### Inline CRTF string ⇒ DataArray with `attrs["creation"]`

In [ ]:
crtf_inline = """
#CRTF
+circle[[120pix,80pix], 40pix]
-rotbox[[120pix,80pix],[36pix,18pix], theta_m=30]
+box[[20pix,160pix],[70pix,190pix]]
""".strip()

mask_inline = select_mask(
    xda, select=crtf_inline
)  # default return_kind: dataarray-dask

print("type:", type(mask_inline).__name__)
print("has creation:", "creation" in mask_inline.attrs)
print("creation:\n", mask_inline.attrs.get("creation"))

### CRTF file (backticked string and `Path`) ⇒ creation reflects how it was referenced

In [ ]:
# Code
crtf_text = """
#CRTF
box[[30pix,40pix],[120pix,140pix]]
""".strip()

crtf_path = Path("example_region.crtf")
_ = crtf_path.write_text(crtf_text, encoding="utf-8")

mask_bt = select_mask(xda, select=f"`{crtf_path.as_posix()}`")
mask_path = select_mask(xda, select=crtf_path)

print("backticked creation:", mask_bt.attrs.get("creation"))
print("Path creation     :", mask_path.attrs.get("creation"))

### Expression over named masks ⇒ `creation` equals the expression string

In [ ]:
# Code
roi = select_mask(
    xda, select="box[[10pix,10pix],[150pix,120pix]]", return_kind="dataarray-numpy"
)
bad = select_mask(
    xda, select="box[[50pix,20pix],[70pix,200pix]]", return_kind="dataarray-numpy"
)

expr = "roi & ~bad"
mask_expr = select_mask(xda, select=expr, mask_source={"roi": roi, "bad": bad})

print("expr creation:", mask_expr.attrs.get("creation"))

### Combining masks with bitwise ops

xarray ops may drop attributes. Use a tiny helper to **merge** the left/right `creation` strings onto the combined mask.

In [ ]:
# Example from the spec
m1 = select_mask(xda, select="box[[1pix,1pix],[200pix,30pix]]")
m2 = select_mask(xda, select="box[[270pix,1pix],[170pix,200pix]]")

# The creation string is lost here
m3 = select_mask(xda, select=(m1 | m2))

# Here is how to preserve the creation string
edge = combine_with_creation(m1, "|", m2)

print("m3 creation\n", m3.attrs.get("creation"))
print("edge creation:\n", edge.attrs.get("creation"))

### Quick sanity: `creation` preserved with different return kinds

Note that because unwrapped np and dask arrays have no attr
functionality, those return types will not have associated creation information. Creation information is only associated with xrDataArrays.

In [ ]:
m_danp = select_mask(
    xda, select="circle[[50pix,50pix], 20pix]", return_kind="dataarray-numpy"
)
m_dadask = select_mask(
    xda, select="circle[[50pix,50pix], 20pix]", return_kind="dataarray-dask"
)
m_np = select_mask(xda, select="circle[[50pix,50pix], 20pix]", return_kind="numpy")
m_dask = select_mask(xda, select="circle[[50pix,50pix], 20pix]", return_kind="dask")

print("DataArray[NumPy] creation:", m_danp.attrs.get("creation"))
print("DataArray[Dask]  creation:", m_dadask.attrs.get("creation"))
print("NumPy has attrs?          ", hasattr(m_np, "attrs"))
print("Dask array has attrs?     ", hasattr(m_dask, "attrs"))

### Suggested pattern for building complex masks with readable provenance

Write small pieces with `select_mask(..., select=...)` so each piece records its own
`creation`; then combine them via `combine_with_creation(...)` to carry a human-readable
recipe along the pipeline.

In [ ]:
# Code
sky_core = select_mask(xda, select="ellipse[[140pix,100pix],[60pix,30pix], theta_m=15]")
sky_edge = select_mask(xda, select="annulus[[140pix,100pix], [30pix, 80pix]]")

sky_mask = combine_with_creation(sky_core, "|", sky_edge)
print(f"Type of sky_mask {type(sky_mask)}")
print(f"Type of array backing the DataArray {type(sky_mask.data)}")
print("sky_mask creation:\n", sky_mask.attrs["creation"])
sky_mask

### `creation_hint` demos

In [ ]:
ny, nx = 120, 160
xda = xr.DataArray(np.zeros((ny, nx), dtype=float), dims=("x", "y"))

### CRTF inline — default creation vs. override with `creation_hint`

In [ ]:
crtf_inline = """
#CRTF
+circle[[60pix,60pix], 30pix]
-rotbox[[60pix,60pix],[36pix,18pix], theta_m=30]
""".strip()

m_crtf_default = select_mask(xda, select=crtf_inline, return_kind="dataarray-dask")
m_crtf_hint = select_mask(
    xda,
    select=crtf_inline,
    return_kind="dataarray-dask",
    creation_hint="CRTF: circle(60,60,30) minus rotbox(60,60,36x18,theta_m=30deg)",
)

print("default creation == text:", m_crtf_default.attrs.get("creation") == crtf_inline)
print("override creation:", m_crtf_hint.attrs.get("creation"))

### CRTF file (backticked/path) — creation is file **contents**, can be overridden

In [ ]:
crtf_text = """
#CRTF
box[[20pix,30pix],[90pix,80pix]]
""".strip()
p = Path("example_region.crtf")
_ = p.write_text(crtf_text, encoding="utf-8")

m_bt = select_mask(xda, select=f"`{p.as_posix()}`", return_kind="dataarray-numpy")
m_path = select_mask(xda, select=p, return_kind="dataarray-numpy")
m_file_hint = select_mask(
    xda,
    select=p,
    return_kind="dataarray-numpy",
    creation_hint="CRTF file: example_region.crtf",
)

print("backticked creation == file text:", m_bt.attrs.get("creation") == crtf_text)
print("path creation == file text:", m_path.attrs.get("creation") == crtf_text)
print("override creation:", m_file_hint.attrs.get("creation"))

### Expression over named masks — expanded creation vs. override

In [ ]:
roi = select_mask(
    xda, select="box[[10pix,10pix],[120pix,90pix]]", return_kind="dataarray-numpy"
)
bad = select_mask(
    xda, select="box[[40pix,20pix],[70pix,40pix]]", return_kind="dataarray-numpy"
)

expr = "roi & ~bad"
m_expr_default = select_mask(
    xda, select=expr, mask_source={"roi": roi, "bad": bad}, return_kind="dataarray-dask"
)
m_expr_hint = select_mask(
    xda,
    select=expr,
    mask_source={"roi": roi, "bad": bad},
    return_kind="dataarray-dask",
    creation_hint="(box[[10pix,10pix],[120pix,90pix]]) & ~(box[[40pix,20pix],[70pix,40pix]])",
)

print("default creation (expanded or expr):", m_expr_default.attrs.get("creation"))
print("override creation:", m_expr_hint.attrs.get("creation"))

## Creation provenance for selections: `creation_hint` and `combine_with_creation`

These cells demonstrate how to attach human-readable provenance to masks created from non-CRTF sources, and how to combine masks while preserving and merging their creation strings.

In [ ]:
nx, ny = 120, 160
xda = xr.DataArray(np.zeros((nx, ny), dtype=float), dims=("x", "y"))

### Non-CRTF masks with `creation_hint`

If a mask originates from an array (NumPy, Dask, or xarray) rather than textual CRTF/expressions, provide a `creation_hint` so downstream results carry reproducible provenance.

In [ ]:
# xarray.DataArray boolean with a custom hint
roi_named = xr.DataArray(np.zeros((nx, ny), dtype=bool), dims=("x", "y"), name="my_roi")
roi_named.values[20:60, 30:90] = True
m_roi = select_mask(
    xda,
    select=roi_named,
    return_kind="dataarray-numpy",
    creation_hint="ROI rect [x:20..59, y:30..89]",
)
print("m_roi.creation:", m_roi.attrs.get("creation"))

# NumPy column (broadcasts across x) with a hint
col = np.zeros((nx, 1), dtype=bool)
print(col.shape)
col[10:25, 0] = True
m_col = select_mask(
    xda,
    select=col,
    return_kind="dataarray-numpy",
    creation_hint="numpy col rows 10..24",
)
print("m_col.creation:", m_col.attrs.get("creation"))

# Dask mask with a hint
dmask = da.random.random((nx, ny), chunks=(40, 40)) > 0.98
m_dask = select_mask(
    xda,
    select=dmask,
    return_kind="dataarray-dask",
    creation_hint="dask random > 0.98 (chunks=40x40)",
)
print("m_dask.creation:", m_dask.attrs.get("creation"))

### Expressions with an optional `creation_hint` override

Expressions already produce a readable `creation` (possibly expanded using named-mask provenance). You can override it explicitly when needed.

In [ ]:
roi = select_mask(
    xda, select="box[[10pix,10pix],[120pix,90pix]]", return_kind="dataarray-numpy"
)
bad = select_mask(
    xda, select="box[[40pix,20pix],[70pix,40pix]]", return_kind="dataarray-numpy"
)

expr = "roi & ~bad"
m_expr_default = select_mask(
    xda, select=expr, mask_source={"roi": roi, "bad": bad}, return_kind="dataarray-dask"
)
print("m_expr_default.creation:", m_expr_default.attrs.get("creation"))

m_expr_hint = select_mask(
    xda,
    select=expr,
    mask_source={"roi": roi, "bad": bad},
    return_kind="dataarray-dask",
    creation_hint="(box[[10pix,10pix],[120pix,90pix]]) & ~(box[[40pix,20pix],[70pix,40pix]])",
)
print("m_expr_hint.creation:", m_expr_hint.attrs.get("creation"))

### `combine_with_creation`: merge masks and provenance

Combine two boolean `DataArray` masks with `|`, `&`, or `^`. The result’s `creation` is built from the inputs’ creation strings.

In [ ]:
m1_src = np.zeros((nx, ny), dtype=bool)
m1_src[20:60, 30:90] = True
m2_src = da.random.random((nx, ny), chunks=(40, 40)) > 0.98

m1 = select_mask(
    xda,
    select=m1_src,
    return_kind="dataarray-numpy",
    creation_hint="numpy rect [x:20..59, y:30..89]",
)
m2 = select_mask(
    xda,
    select=m2_src,
    return_kind="dataarray-dask",
    creation_hint="dask random > 0.98 (chunks=40x40)",
)

combo_or = combine_with_creation(m1, "|", m2, return_kind="dataarray-dask")
combo_and = combine_with_creation(m1, "&", m2, return_kind="dataarray-dask")
combo_xor = combine_with_creation(m1, "^", m2, return_kind="dataarray-dask")

print("OR creation :", combo_or.attrs.get("creation"))
print("AND creation:", combo_and.attrs.get("creation"))
print("XOR creation:", combo_xor.attrs.get("creation"))

### Preserving provenance when changing return kind

Coerce a combined mask to NumPy-backed `DataArray` while preserving the same `creation`.

In [ ]:
combo_np = select_mask(
    xda,
    select=combo_or,
    return_kind="dataarray-numpy",
    creation_hint=combo_or.attrs.get("creation"),
)
print("numpy-backed:", not hasattr(combo_np.data, "chunks"))
print("creation    :", combo_np.attrs.get("creation"))

## Combine CRTF file-based masks with inline CRTF and with non-CRTF boolean masks (preserving `creation`)

In [ ]:
# Demo image
nx, ny = 200, 240
xda = xr.DataArray(np.zeros((nx, ny), dtype=float), dims=("x", "y"))

### Build two CRTF masks
- **File-based**: created from a backticked filename; its `creation` is the file contents.
- **Inline**: CRTF text in a Python string; its `creation` is that exact string.

In [ ]:
# File-based CRTF
crtf_file_text = """
#CRTF
+box[[30pix,40pix],[120pix,140pix]]
-rotbox[[75pix,90pix],[30pix,15pix], theta_m=20]
""".strip()

crtf_path = Path("example_region.crtf")
_ = crtf_path.write_text(crtf_file_text, encoding="utf-8")

m_file = select_mask(
    xda, select=f"`{crtf_path.as_posix()}`", return_kind="dataarray-dask"
)

print(
    "file-based creation equals file contents:",
    m_file.attrs.get("creation") == crtf_file_text,
)

# Inline CRTF
crtf_inline_text = """
#CRTF
+circle[[160pix,120pix], 45pix]
-ellipse[[160pix,120pix],[60pix,24pix], theta_m=35]
""".strip()

m_inline = select_mask(xda, select=crtf_inline_text, return_kind="dataarray-dask")

print(
    "inline creation equals inline text:",
    m_inline.attrs.get("creation") == crtf_inline_text,
)

### Combine **file-based** and **inline** CRTF masks

We expect the combined mask’s `creation` to be:
({file_crtf_text}) | ({inline_crtf_text})


In [ ]:
m_file_or_inline = combine_with_creation(
    m_file, "|", m_inline, return_kind="dataarray-dask"
)
print("combined creation:\n", m_file_or_inline.attrs.get("creation"))

# Optional sanity check that the provenance string matches the
# exact inputs
expected_creation = f"({crtf_file_text}) | ({crtf_inline_text})"
print(
    "creation matches expectation:",
    m_file_or_inline.attrs.get("creation") == expected_creation,
)

### Create a **non-CRTF** boolean mask with a `creation_hint`

Here we use a NumPy rectangle and a Dask random-threshold mask to show both styles; pick either for combining.

In [ ]:
# NumPy rectangle (non-CRTF), with a clear creation_hint
np_rect = np.zeros((nx, ny), dtype=bool)
np_rect[20:60, 150:210] = True
m_np_rect = select_mask(
    xda,
    select=np_rect,
    return_kind="dataarray-numpy",
    creation_hint="numpy rect [x:20..59, y:150..209]",
)
print("np_rect creation:", m_np_rect.attrs.get("creation"))

# Dask random mask (non-CRTF), with a creation_hint
dmask = da.random.random((nx, ny), chunks=(50, 50)) > 0.985
m_dask_rand = select_mask(
    xda,
    select=dmask,
    return_kind="dataarray-dask",
    creation_hint="dask random > 0.985 (chunks=50x50)",
)
print("dask_rand creation:", m_dask_rand.attrs.get("creation"))

### Combine the **CRTF (file|inline)** mask with a **non-CRTF** mask

We expect the combined `creation` to preserve both sides:
(({file_crtf_text}) | ({inline_crtf_text})) & (non_crtf_creation)

You can use `|`, `&`, or `^` as needed.

In [ ]:
# Use the NumPy rectangle mask
a = m_file_or_inline & m_np_rect
m_all = combine_with_creation(
    m_file_or_inline, "&", m_np_rect, return_kind="dataarray-dask"
)
print("combined-with-nonCRTF creation:\n", m_all.attrs.get("creation"))

expected_all = (
    f"(({crtf_file_text}) | ({crtf_inline_text})) & ({m_np_rect.attrs.get('creation')})"
)
print("creation matches expectation:", m_all.attrs.get("creation") == expected_all)

In [ ]:
# Alternatively, use the Dask random mask
m_all2 = combine_with_creation(
    m_file_or_inline, "|", m_dask_rand, return_kind="dataarray-dask"
)
print("combined-with-dask creation:\n", m_all2.attrs.get("creation"))

expected_all2 = f"(({crtf_file_text}) | ({crtf_inline_text})) | ({m_dask_rand.attrs.get('creation')})"
print("creation matches expectation:", m_all2.attrs.get("creation") == expected_all2)

### (Optional) Coerce to NumPy-backed while **preserving** `creation`

If you need a NumPy-backed `DataArray`, rewrap with `select_mask(..., return_kind="dataarray-numpy")`,
passing through the same `creation`.

In [ ]:
m_all_np = select_mask(
    xda,
    select=m_all,
    return_kind="dataarray-numpy",
    creation_hint=m_all.attrs.get("creation"),
)
print("numpy-backed:", not hasattr(m_all_np.data, "chunks"))
print(
    "creation preserved:", m_all_np.attrs.get("creation") == m_all.attrs.get("creation")
)